In [1]:
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime
from openai import OpenAI
from PIL import Image
import base64
from pdf2image import convert_from_path
from io import BytesIO
import re
import google.generativeai as genai
import PIL.Image
import mysql.connector
import base64
import smtplib
import time
import os
import locale
import requests
import tempfile

# Função para converter a imagem para base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Código para converter o PDF em base 64  
def convert_pdf_to_jpg_base64(pdf_path):
    # Converte PDF em imagens
    pages = convert_from_path(pdf_path, poppler_path=r'C:\poppler\bin')  # Atualize com o caminho do Poppler no Windows
    base64_images = []

    # Converte cada página para JPG e, em seguida, para base64
    for page in pages:
        buffered = BytesIO()
        page.save(buffered, format="JPEG")
        img_str = base64.b64encode(buffered.getvalue()).decode('utf-8')
        base64_images.append(img_str)

    return base64_images

def convert_pdf_to_images(pdf_path):
    # Cria uma pasta temporária
    temp_dir = tempfile.mkdtemp()

    images = convert_from_path(pdf_path, poppler_path=r'C:\poppler\bin')
    
    image_paths = []
    for i, image in enumerate(images):
        # Define o caminho da imagem na pasta temporária
        image_path = os.path.join(temp_dir, f"page_{i + 1}.jpg")
        # Salva a imagem
        image.save(image_path, 'JPEG')
        image_paths.append(image_path)
    
    return image_paths
  
# Função que chama o Gemini
def executa_gemini(caminho_arquivo):
    api_key = "'SECRET_REMOVED_BY_AI'"

    GOOGLE_API_KEY=(api_key)

    genai.configure(api_key=GOOGLE_API_KEY)

    image_paths = convert_pdf_to_images(caminho_arquivo)

    # Definições da geração do modelo
    generation_config = {
        "candidate_count": 1, # Quantidade de respostas
        "temperature": 0 # Temperatura do modelo para gerar um texto, vai até 1, quanto maior, mais criativo o modelo vai ser
    }

    # Definições de segurança
    safety_settings = {
        "HARASSMENT": "BLOCK_NONE", # Bloqueios
        "HATE": "BLOCK_NONE",
        "SEXUAL": "BLOCK_NONE",
        "DANGEROUS": "BLOCK_NONE"
    }

    model = genai.GenerativeModel(model_name='gemini-pro-vision', generation_config=generation_config, safety_settings=safety_settings)

    results = []
    for image_path in image_paths:
        # Usa a primeira imagem para enviar para a API
        img_path = image_path
        
        # Leitura da imagem para PIL
        img = Image.open(img_path)

        response = model.generate_content(["Me retorne: número do pedido (ou cotação), CNPJ, data da cotação e a tabela completa (todas as informações) de todos os itens cotados, separados por cotação. Um ponto importante: Podemos ter mais de uma cotação nesta imagem, quero que você extraia todos os pedidos, incluside de todas as páginas. Separe da seguinte forma: Pedido: x | CNPJ: xx.xxx.xxx/xxxx-xx | Data da cotação: xx/xx/xxxx | Código de barras: x | Fabricante: x | Preço de compra: x | Desconto: x | Quantidade: x | Total: x. Essas informações devem formar uma linha SEMPRE, o mesmo pedido pode estar numa página e o restante em outro, você deve perceber isso e fazer a unificação, reforçando, você é obrigado a manter esse padrão: Pedido: x | CNPJ: xx.xxx.xxx/xxxx-xx | Data da cotação: xx/xx/xxxx | Código de barras: x | Fabricante: x | Preço de compra: x | Desconto: x | Quantidade: x | Total: x. SEMPRE RETORNE NESSE MODELO: Pedido: x | CNPJ: xx.xxx.xxx/xxxx-xx | Data da cotação: xx/xx/xxxx | Código de barras: x | Fabricante: x | Preço de compra: x | Desconto: x | Quantidade: x | Total: x", img], stream=True)

        # Adiciona o resultado ao conjunto de resultados
        if response.candidates:
            results.append(response.resolve()[0]['text'])

    # Combina todos os resultados em um único texto
    combined_results = "\n".join(results)
    return combined_results

def executa_gpt(caminho_arquivo):
    base64_images = convert_pdf_to_jpg_base64(pdf_path=caminho_arquivo)
    api_key = "'SECRET_REMOVED_BY_AI'"
    client = OpenAI(api_key=api_key)
    
    results = []
    for base64_image in base64_images:
        print(base64_image)
        # Envia a solicitação para o GPT para cada imagem
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Vou te mandar um arquivo de um pedido, é composto por uma tabela com os itens, em que temos um cabeçalho com o número do pedido, cotação do cliente, CNPJ e, logo abaixo, uma tabela com os itens, quero que você me retorne: número do pedido (ou cotação), CNPJ, data da cotação e a tabela completa (todas as informações) de todos os itens cotados. Um ponto importante: Essas informações devem formar uma linha SEMPRE, pode acontecer de o pedido estar em uma página e o restante, desse mesmo pedido, em outro, você deve perceber isso e fazer a unificação para SEMPRE ficar nesse padrão: Pedido: x | CNPJ: xx.xxx.xxx/xxxx-xx | Data da cotação: xx/xx/xxxx | Código de barras: x | Fabricante: x | Preço de compra: x | Desconto: x | Quantidade: x | Total: x."},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}", # A saída da imagem deve ser um .jpg
                            },
                        },
                    ],
                }
            ],
            # max_tokens=4096,
        )
        # Adiciona o resultado ao conjunto de resultados
        results.append(response.choices[0].message.content)
    
    # Combina todos os resultados em um único texto
    combined_results = "\n".join(results)
    return combined_results

def extract_information(text):
    # Define um padrão regex para capturar o texto após os dois pontos de cada item
    pattern = r"Pedido:\s*([^\|]+)\s*\|\s*CNPJ:\s*([^\|]+)\s*\|\s*Data da cotação:\s*([^\|]+)\s*\|\s*Código de barras:\s*([^\|]+)\s*\|\s*Fabricante:\s*([^\|]+)\s*\|\s*Preço de compra:\s*([^\|]+)\s*\|\s*Desconto:\s*([^\|]+)\s*\|\s*Quantidade:\s*([^\|]+)\s*\|\s*Total:\s*([^\|]+)"

    # Encontra todas as correspondências no texto
    matches = re.findall(pattern, text)

    # Converte as correspondências em uma lista de dicionários
    extracted_data = []
    for match in matches:
        data = {
            "Pedido": match[0].strip(),
            "CNPJ": match[1].strip(),
            "Data da cotação": match[2].strip(),
            "Código de barras": match[3].strip(),
            "Fabricante": match[4].strip(),
            "Preço de compra": match[5].strip(),
            "Desconto": match[6].strip(),
            "Quantidade": match[7].strip(),
            "Total": match[8].strip()
        }
        extracted_data.append(data)

    return extracted_data

#texto = executa_gemini(caminho_arquivo=r"C:\Users\Nicolas\Downloads\335511.pdf")
#texto = executa_gemini(caminho_arquivo=r"C:\Users\Nicolas\Downloads\BPM TLV\PEDIDO SOLAR COTY GAM 22-09 (1).pdf")
texto = executa_gpt(caminho_arquivo=r"C:\Users\Nicolas\Downloads\BPM TLV\PEDIDO SOLAR COTY GAM 22-09 (1).pdf")
lista_texto_dividido = texto.split("\n")

for texto_div in lista_texto_dividido:
    if texto_div != "" or texto_div != None:
        texto_extraido = extract_information(text=texto_div)

        lista_texto_validar = texto_div.split("|")
        for texto in lista_texto_validar:
            pedido, cnpj, data_cotacao, codigo_barra, fabricante, preco_compra, desconto, quantidade, total = texto

#print(texto)

c:\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


...
